In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType
from pyspark.sql.functions import (
    col, concat_ws, count, when, avg, mean, stddev, lit, element_at, sum as spark_sum
)
from pyspark.ml.feature import StandardScaler, VectorAssembler, PCA
from pyspark.ml.stat import Correlation
from pyspark.ml.functions import vector_to_array

In [ ]:
color_1 = '#27AAE2'
color_2 = '#2B3990'

In [ ]:
drive.mount('/content/drive')

In [ ]:
# !pip install pyspark

In [ ]:
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("eda analysis") \
    .getOrCreate()

In [ ]:
file_path = '/content/drive/MyDrive/DDAM project/data/df.csv'

In [ ]:
df = ss.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
tile_mapping = {
    '18QWF': 'La Gonave', '18QYF': 'La Gonave', '18QYG': 'La Gonave',
    '48MXU': 'Jakarta', '48MYU': 'Jakarta',
    '16PCC': 'Motagua',
    '16PDC': 'Ulua',
    '16PEC': 'La Ceiba',
    '16QED': 'Roatan',
    '19QDA': 'Santo Domingo',
    '30VWH': 'Isle of May',
    '36JUN': 'Durban',
    '48PZC': 'Danang',
    '50LLR': 'Bali',
    '51PTS': 'Manila',
    '51RVQ': 'Yangtze',
    '52SDD': 'Nakdong'
}

df = df.replace(to_replace=tile_mapping, subset=['Tile'])

In [ ]:
df.show()

# **Statistical Analysis**

In [ ]:
df.columns

In [ ]:
statistics = df.describe()
statistics.show()

# **Class**

In [ ]:
count_class = df.groupBy('class').count()

tot = df.count()

count_class = count_class.withColumn('perc', (col('count') / tot) * 100)

count_class = count_class.orderBy(col('count').desc())

count_class.show()

In [ ]:
# To Pandas for the plot
count_class_pd = count_class.toPandas()

sns.set_theme(style='whitegrid', context='talk')

plt.figure(figsize=(12, 7))

ax = sns.barplot(x='class', y='count', data=count_class_pd, color='#2B3990',
                 edgecolor='black', linewidth=1)

for i, container in enumerate(ax.containers):
    labels = [f'{val:.1f}%' for val in count_class_pd['perc']]
    ax.bar_label(container, labels=labels, fontsize=12, fontweight='bold')

ax.set_title('Pixel percentage distribution by Class',
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Target Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count', fontsize=14, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
df_prep = df.withColumn('patch_id',
                        concat_ws('_',
                                 col('Date').cast('string'),
                                 col('Tile').cast('string'),
                                 col('Image').cast('string')))

df_prep = df_prep.select('patch_id', 'Class')


In [ ]:
# Count pixels for each class
binary_spark = df_prep.groupBy('patch_id') \
                      .pivot('Class') \
                      .count() \
                      .fillna(0)

class_cols = [c for c in binary_spark.columns if c != 'patch_id']

for c in class_cols:
    binary_spark = binary_spark.withColumn(c, when(col(c) > 0, 1).otherwise(0))

# To Pandas for the final matrix
binary_matrix = binary_spark.toPandas()
binary_matrix.set_index('patch_id', inplace=True)

# Co - occurrence
co_occurrence = binary_matrix.T.dot(binary_matrix)

# Normalization
co_occurrence_pct = co_occurrence.div(co_occurrence.values.diagonal(), axis=0) * 100

In [ ]:
# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(co_occurrence_pct,
            annot=False,
            fmt='.2f',
            cmap='RdYlBu_r',
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'label': 'Conditional Co-occurrence (%)'},
            vmin=0,
            vmax=100)

plt.title('Percentage of Class Co-occurrence',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Class', fontsize=14, fontweight='bold')
plt.ylabel('Class', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
assembler_class_corr = VectorAssembler(inputCols=class_cols, outputCol="class_features")
binary_vector_df = assembler_class_corr.transform(binary_spark).select("class_features")

# Correlation matrix
matrix_class = Correlation.corr(binary_vector_df, 'class_features').head()
pearson_matrix_class = matrix_class[0].toArray()

pearson_corr = pd.DataFrame(
    pearson_matrix_class,
    columns=class_cols,
    index=class_cols
)

plt.figure(figsize=(20, 16))
sns.heatmap(pearson_corr,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'label': 'Pearson Correlation (Phi Coefficient)'},
            vmin=-1,
            vmax=1)

plt.title('Pearson Correlation Between Classes\n(Symmetric)',
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Class', fontsize=14, fontweight='bold')
plt.ylabel('Class', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()
plt.show()

# **City**

In [ ]:
count_city = df.groupBy('Tile').count()

tot_city = df.count()
count_city = count_city.withColumn('perc', (col('count') / tot_city) * 100).orderBy(col('count').desc())

# To pandas for the plot
count_city_pd = count_city.toPandas()

sns.set_theme(style='whitegrid', context='talk')

# Barplot
plt.figure(figsize=(12, 7))
ax = sns.barplot(x='Tile', y='count', data=count_city_pd, color=color_1,
                 edgecolor='black', linewidth=1)

for i, container in enumerate(ax.containers):
    labels = [f'{val:.1f}%' for val in count_city_pd['perc']]
    ax.bar_label(container, labels=labels, fontsize=12, fontweight='bold')

ax.set_title('Pixel percentage distribution by City',
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('City', fontsize=14, fontweight='bold')
ax.set_ylabel('Count', fontsize=14, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

# **Confidence**

In [ ]:
count_conf = df.groupBy('Confidence').count()

tot = df.count()

count_conf = count_conf.withColumn('perc', (col('count') / tot) * 100)

count_conf = count_conf.orderBy(col('count').desc())

count_conf.show()

In [ ]:
# Filter dataframe, only Marine Debris
df_marine_debris = df.filter(col("class") == "Marine Debris")

df_marine_debris_g = df_marine_debris.groupBy('Confidence').count()

tot = df_marine_debris.count()

df_marine_debris_p = df_marine_debris_g.withColumn('perc', (col('count') / tot) * 100)

df_marine_debris_p = df_marine_debris_p.orderBy(col('count').desc())

df_marine_debris_p.show()



In [ ]:
ct_spark = df.filter(col("class") == "Marine Debris") \
             .withColumnRenamed('Tile', 'City') \
             .groupBy('City') \
             .pivot('Confidence') \
             .count() \
             .fillna(0)

ct = ct_spark.toPandas()

ct.set_index('City', inplace=True)

ct['Total'] = ct.sum(axis=1)
ct = ct.sort_values('Total', ascending=False)
ct = ct.drop(columns=['Total'])

ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

In [ ]:
sns.set_theme(style='whitegrid', context='talk')

confidence_levels = ['Low', 'Moderate', 'High']
confidence_colors = {
    'Low': '#c6dbef',
    'Moderate': color_1,
    'High': color_2
}
colors = [confidence_colors[level] for level in confidence_levels]

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

left = np.zeros(len(ct_pct))
for idx, conf_level in enumerate(confidence_levels):
    values = ct_pct[conf_level]

    ax.barh(ct_pct.index, values, left=left,
            color=colors[idx],
            edgecolor='black',
            linewidth=1.5,
            label=f'Confidence: {conf_level}')

    for i, (city, val) in enumerate(zip(ct_pct.index, values)):
        if val > 3:
            x_pos = left[i] + val / 2
            ax.text(x_pos, i, f'{val:.1f}%',
                   ha='center', va='center',
                   fontsize=14, fontweight='bold',
                   color='white')

    left += values

ax.set_title('Marine Debris: Confidence Distribution by City',
             fontsize=22, fontweight='bold', pad=20)
ax.set_xlabel('Percentage (%)', fontsize=18, fontweight='bold')
ax.set_ylabel('City', fontsize=18, fontweight='bold')
ax.set_xlim(0, 100)

ax.legend(title='Confidence Level', loc='lower right', fontsize=12)

ax.tick_params(axis='both', labelsize=14)

sns.despine()

plt.tight_layout()
plt.show()

# **Missing Values**

In [ ]:
nan_expressions = [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]

df.select(*nan_expressions).show()

# **Spectral Bands**

In [ ]:
COLOR_1     = '#27AAE2'
COLOR_2     = '#2B3990'
COLOR_RED   = '#E8403A'
COLOR_WHITE = '#FFFFFF'

cmap_paper = LinearSegmentedColormap.from_list(
    "paper_diverging",
    [COLOR_2, COLOR_1, COLOR_WHITE, COLOR_RED],
    N=512
)

features_to_plot = ["NDVI", "FDI", "NDWI", "NRD", "CON", "HOMO", "ASM"]

assembler = VectorAssembler(inputCols=features_to_plot, outputCol="features_vec", handleInvalid="skip")
df_vec = assembler.transform(df)

scaler = StandardScaler(inputCol="features_vec", outputCol="scaled_features_vec", withStd=True, withMean=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)

df_scaled = df_scaled.withColumn("scaled_array", vector_to_array(col("scaled_features_vec")))

for i, feature in enumerate(features_to_plot):
    df_scaled = df_scaled.withColumn(f"scaled_{feature}", element_at(col("scaled_array"), i + 1))

agg_exprs = [avg(f"scaled_{feature}").alias(feature) for feature in features_to_plot]
df_means_scaled = df_scaled.groupBy("class").agg(*agg_exprs)

df_scaled_pd = df_means_scaled.toPandas()
df_scaled_pd = df_scaled_pd.sort_values("class").set_index("class")

fig, ax = plt.subplots(figsize=(13, 9))

fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#F7F9FC')

sns.heatmap(
    df_scaled_pd,
    ax=ax,
    cmap=cmap_paper,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 14},
    linewidths=0.6,
    linecolor='#DDDDDD',
    cbar_kws={
        'label': 'Z-Score',
        'shrink': 0.75,
        'aspect': 25,
        'pad': 0.02,
    },
    vmin=-2.5,
    vmax=2.5,
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=14, colors='#1A1A2E')
cbar.set_label('Z-Score', fontsize=18, fontweight='bold',
               color='#1A1A2E', labelpad=10)
cbar.outline.set_edgecolor('#CCCCCC')

ax.set_xlabel("Features", fontsize=18, fontweight='bold',
              color='#1A1A2E', labelpad=12)
ax.set_ylabel("Class", fontsize=18, fontweight='bold',
              color='#1A1A2E', labelpad=12)

ax.tick_params(axis='x', labelsize=16, colors='#1A1A2E', length=0)
ax.tick_params(axis='y', labelsize=16, colors='#1A1A2E', length=0)
plt.xticks(rotation=0)
plt.yticks(rotation=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('#CCCCCC')
    spine.set_linewidth(0.8)

plt.tight_layout(pad=2.0)
plt.savefig("zscore_heatmap_pyspark.png", dpi=200,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


# **Correlation**

In [ ]:
numeric_features = [
    c.name for c in df.schema
    if isinstance(c.dataType, (DoubleType, IntegerType, LongType, FloatType))
]

assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features"
)

df_vector = assembler.transform(df).select("features")

In [ ]:
matrix = Correlation.corr(df_vector, 'features').head()

correlation_matrix = matrix[0].toArray()

In [ ]:
corr_df = pd.DataFrame(
    correlation_matrix,
    columns=numeric_features,
    index=numeric_features
)

fig, ax = plt.subplots(figsize=(36, 20))
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#F7F9FC')

sns.heatmap(
    corr_df,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap=cmap_paper,
    linewidths=.5,
    cbar=True
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=20, colors='#1A1A2E')
cbar.set_label('Pearson Correlation', fontsize=20, fontweight='bold',
               color='#1A1A2E', labelpad=10)
cbar.outline.set_edgecolor('#CCCCCC')

ax.tick_params(axis='x', labelsize=20, colors='#1A1A2E', length=0)
ax.tick_params(axis='y', labelsize=20, colors='#1A1A2E', length=0)
plt.xticks(rotation=45)
plt.yticks(rotation=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('#CCCCCC')
    spine.set_linewidth(0.8)

plt.tight_layout(pad=2.0)
plt.savefig("correlation_heatmap.png", dpi=200,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


# **PCA for dimensionality reduction**

**Due to the fact that spectral variables are higly correlated and our next steps are clustering and ML with explainable AI, it's chosen to do PCA on these features. So new indipendent rapresentative variables are obtained**

In [ ]:
spectral_cols = [
    "nm440", "nm490", "nm560", "nm665", "nm705",
    "nm740", "nm783", "nm842", "nm865", "nm1600", "nm2200"
]

K_COMPONENTS = 1

assembler = VectorAssembler(
    inputCols=spectral_cols,
    outputCol="spectral_features",
    handleInvalid="skip"
)

df_vector = assembler.transform(df)

# pca method
pca = PCA(k=len(spectral_cols), inputCol="spectral_features", outputCol="pca_vector")

# fit
model = pca.fit(df_vector)

df_pca_result = model.transform(df_vector)

# Explained variance
cumulative_variance = model.explainedVariance.cumsum()

for i, var in enumerate(cumulative_variance):
    print(f"PC_{i+1} Cumulative variance: {var*100:.2f}%")

In [ ]:
df_pca_result = df_pca_result.withColumn(
    "pca_values_array",
    vector_to_array(col("pca_vector"))
)

for i in range(K_COMPONENTS):
    df_pca_result = df_pca_result.withColumn(
        f"PC_{i+1}",
        element_at(col("pca_values_array"), i + 1)
    )

cols_to_keep = [
    c for c in df.columns
    if c not in spectral_cols and c not in ['SI', 'HOMO', 'ASM', 'FAI']
]
pca_cols = [f"PC_{i+1}" for i in range(K_COMPONENTS)]

df_final = df_pca_result.select(cols_to_keep + pca_cols)

df_final.show(5)

**From explained variance of components only one component can be used.  
Also 'SI' can be eliminated due to the high negative correlation with all spectral features.
Also 'HOMO', 'FAI, 'ASM'**

# **PCA Dataset Saving**

In [ ]:
output_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_finale.csv"

df_final.write.csv(output_path, header=True, mode="overwrite", sep=",")